In [1]:
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex

from src.utils import formatTimedelta, parallelizeFunction, time2localtime

In [2]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "BAJA",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ALTA",
    # "PlatformForecast": "PREDICCIÓN",
    # "Stopped": "STOP",
    # "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "APROXIMACIÓN",
            "MANIOBRALLEGADA",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "EXIT",
            "MANIOBRASALIDA",
            "MANIOBRA",
        ]
    )
}

In [3]:
fname = Path(
    r"C:\Users\jose.espinosa\Documents\Data\xsiv\PRO\2024-05\graylog 2024-05-27-100000 - 2024-05-28-100000.csv"
)

In [4]:
# Leemos el fichero y usamos solo los audited
with fname.open("r", encoding="utf8") as f:
    lines = f.read()
    if regex.search('^"timestamp', lines):
        lines = lines.split("\n", 1)[-1]
    lines = regex.sub(r'(?<!=)(?<=.+)""|(?<==)""(?=.+(?<!=)"")', '"', lines)
    lines = lines.split("\n")

In [5]:
logs = []
cols = ["timestamp_msg", "timestamp_ctc", "Código", "Nombre", "Vía", "Movimiento"]
# for el in lines:
#     if "AUDITED" in el:
#         logs.append(
#             [
#                 regex.search(r'(?<=,").+?(?=#)', el).group(),
#                 regex.search(r'(?<=timestamp=").+?(?=")', el).group(),
#                 regex.search(r'(?<=pointCode=").+?(?=")', el).group(),
#                 regex.search(r'(?<=pointName=").+?(?=")', el).group(),
#                 regex.search(r'(?<=" platformCode=").+?(?=".+?>)', el).group(),
#             ]
#         )


def procLine(line):
    if "AUDITED" in line:
        timestamp_msg = regex.search(r'(?<=,").+?(?=#)', line)
        timestamp_ctc = regex.search(r'(?<=timestamp=").+?(?=")', line)
        Código = regex.search(r'(?<=pointCode=").+?(?=")', line)
        Nombre = regex.search(r'(?<=pointName=").+?(?=")', line)
        Vía = regex.search(r'(?<=" platformCode=").+?(?=".+?>)', line)
        Movimiento = regex.search(r"(?<=<train).+?(?=\s)", line)
        timestamp_msg = timestamp_msg.group() if timestamp_msg else None
        timestamp_ctc = timestamp_ctc.group() if timestamp_ctc else None
        Código = Código.group() if Código else None
        Nombre = Nombre.group() if Nombre else None
        Vía = Vía.group() if Vía else None
        Movimiento = Movimiento.group() if Vía else None
        return [timestamp_msg, timestamp_ctc, Código, Nombre, Vía, Movimiento]


logs = parallelizeFunction(
    procLine,
    data=lines,
    show_progress=True,
    leave=True,
    desc="Leyendo logs",
    output="series",
)
logs = [log for log in logs if log is not None]

Leyendo logs:   0%|          | 0/589408 [00:00<?, ?it/s]

In [6]:
df = pd.DataFrame(logs, columns=cols)

In [9]:
# # df["timestamp_msg"] = parallelizeFunction(
# #     lambda x: pd.to_datetime(x, utc=True).round(freq="s").tz_localize(None),
# #     data=df["timestamp_msg"],
# #     show_progress=True,
# #     leave=True,
# #     desc="Formateando fechas.",
# #     output="series",
# # )
# # df["timestamp_ctc"] = parallelizeFunction(
# #     time2localtime,
# #     data=df["timestamp_ctc"],
# #     show_progress=True,
# #     leave=True,
# #     desc="Formateando fechas.",
# #     output="series",
# # )
# df["timestamp_msg"] = pd.concat(
#     parallelizeFunction(
#         lambda x: x.apply(
#             lambda el: pd.to_datetime(el, utc=True).round(freq="s").tz_localize(None)
#         ),
#         data=np.array_split(
#             df["timestamp_msg"],
#             len(df) // np.min((len(df), 1000)),
#         ),
#         show_progress=True,
#         desc="Formateando fechas.",
#         output="series",
#     )
# )
# df["timestamp_ctc"] = pd.concat(
#     parallelizeFunction(
#         lambda x: x.apply(time2localtime),
#         data=np.array_split(
#             df["timestamp_ctc"],
#             len(df) // np.min((len(df), 1000)),
#         ),
#         show_progress=True,
#         desc="Formateando fechas.",
#         output="series",
#     )
# )

c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


Formateando fechas.:   0%|          | 0/589 [00:00<?, ?it/s]

Formateando fechas.:   0%|          | 0/589 [00:00<?, ?it/s]

In [10]:
df["tdiff"] = (df["timestamp_msg"] - df["timestamp_ctc"]).dt.total_seconds()
df["Diferencia"] = df["tdiff"].apply(formatTimedelta)

In [23]:
df[(df["Nombre"] == "CERCEDA-MEIRAMA") & (df["tdiff"] > 600)]

,timestamp_msg,timestamp_ctc,Código,Nombre,Vía,Movimiento,tdiff,Diferencia
7809,2024-05-27 17:49:25,2024-05-27 17:35:48,31416,CERCEDA-MEIRAMA,2,Departure,817.0,00:13:37
14047,2024-05-27 20:44:07,2024-05-27 20:30:30,31416,CERCEDA-MEIRAMA,2,Departure,817.0,00:13:37
26180,2024-05-28 07:43:22,2024-05-28 07:29:40,31416,CERCEDA-MEIRAMA,2,Departure,822.0,00:13:42
48351,2024-05-28 08:41:56,2024-05-28 08:28:13,31416,CERCEDA-MEIRAMA,1,Departure,823.0,00:13:43
67231,2024-05-27 12:09:13,2024-05-27 11:55:38,31416,CERCEDA-MEIRAMA,2,Departure,815.0,00:13:35
120186,2024-05-27 22:02:24,2024-05-27 21:48:46,31416,CERCEDA-MEIRAMA,2,Departure,818.0,00:13:38
125011,2024-05-27 20:06:00,2024-05-27 19:52:22,31416,CERCEDA-MEIRAMA,1,Departure,818.0,00:13:38
138335,2024-05-27 13:34:14,2024-05-27 13:20:38,31416,CERCEDA-MEIRAMA,2,Departure,816.0,00:13:36
143572,2024-05-28 08:50:55,2024-05-28 08:37:14,31416,CERCEDA-MEIRAMA,2,Departure,821.0,00:13:41
155555,2024-05-27 17:55:54,2024-05-27 17:42:16,31416,CERCEDA-MEIRAMA,2,Departure,818.0,00:13:38


In [26]:
df[df["tdiff"] > 600]["Nombre"].value_counts().head(50)

Nombre
OURENSE                         79
CERCEDA-MEIRAMA                 29
ORDES                           23
UXES                            20
BIF. SAN AMARO                  20
VILAGARCIA DE AROUSA            17
O BURGO-SANTIAGO                15
A FRIELA-MASIDE (APD)           15
PADRON BARBANZA                 14
LOUREDO-VALOS                   13
TABOADELA                       13
OZA DOS RIOS                    12
TUI                             10
SANTIAGO DE COMPOSTELA          10
ARCADE (APD)                     9
ELVIÑA-UNIVERSIDADE (APD)        9
PONTEVEDRA                       7
BAAMONDE                         7
GUITIRIZ                         7
SELA (APD)                       7
TEIXEIRO                         7
LAXOSA                           6
LUGO MERCANCIAS                  6
RUBIAN                           6
LUGO                             6
CANTERA DE CAMPOMARZO            6
PONTE TABOADA                    6
O CARBALLIÑO                     5
POUSA-CRECENT

In [24]:
df[df["tdiff"] > 600]

,timestamp_msg,timestamp_ctc,Código,Nombre,Vía,Movimiento,tdiff,Diferencia
12,2024-05-27 21:52:17,2024-05-27 21:38:38,31312,VEDRA-RIVADULLA,1,Departure,819.0,00:13:39
1872,2024-05-27 21:05:22,2024-05-27 20:51:38,22100,OURENSE,3B,Arrival,824.0,00:13:44
2078,2024-05-27 20:59:35,2024-05-27 20:45:57,22100,OURENSE,3B,Departure,818.0,00:13:38
3836,2024-05-28 08:40:28,2024-05-28 08:26:45,31309,PONTE TABOADA,1,Departure,823.0,00:13:43
4010,2024-05-27 20:56:23,2024-05-27 20:42:45,31306,O IRIXO,1,Departure,818.0,00:13:38
...,...,...,...,...,...,...,...,...
580505,2024-05-27 21:07:51,2024-05-27 20:54:07,22100,OURENSE,3B,Arrival,824.0,00:13:44
581436,2024-05-27 21:08:00,2024-05-27 20:54:16,22100,OURENSE,3B,Arrival,824.0,00:13:44
581615,2024-05-27 12:56:34,2024-05-27 12:42:59,31314,CANTERA DE CAMPOMARZO,1,Departure,815.0,00:13:35
583395,2024-05-27 17:37:51,2024-05-27 17:24:14,B2301,BIF. SAN AMARO,2,PlatformForecast,817.0,00:13:37
